In [1]:
import os
import re
import pandas as pd
from tqdm import tqdm

In [2]:
!git clone -b main https://github.com/AsyaPradana/CBR-Narkotika-PN-Tulungagung.git

Cloning into 'CBR-Narkotika-PN-Tulungagung'...
remote: Enumerating objects: 73, done.
remote: Counting objects: 100% (73/73), done.
remote: Compressing objects: 100% (54/54), done.
remote: Total 73 (delta 15), reused 64 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (73/73), 575.51 KiB | 9.43 MiB/s, done.
Resolving deltas: 100% (15/15), done.


In [3]:
!find /content/CBR-Narkotika-PN-Tulungagung/data/raw -name "*.txt" | wc -l

50


In [4]:
RAW_PATH = "/content/CBR-Narkotika-PN-Tulungagung/data/raw"
OUTPUT_PATH = "/content/CBR-Narkotika-PN-Tulungagung/data/processed"

In [5]:
def extract_metadata(text):

    metadata = {}

    # Nomor perkara
    nomor = re.search(
        r'nomor\s+([\d]+[\w\s\/\-\.]+pn\s+\w+)',
        text,
        re.IGNORECASE
    )

    metadata["no_perkara"] = nomor.group(1).strip() if nomor else ""

    # Pasal
    pasal = re.findall(
        r'pasal\s+\d+\s*(?:ayat\s*\(\d+\))?',
        text,
        re.IGNORECASE
    )

    metadata["pasal"] = "; ".join(set(pasal))

    # Nama terdakwa
    terdakwa = re.search(
        r'nama lengkap\s+(.*?)\s+tempat lahir',
        text,
        re.IGNORECASE
    )

    metadata["terdakwa"] = (
        terdakwa.group(1).strip()
        if terdakwa else ""
    )

    return metadata

In [6]:
def extract_key_content(text):

    hasil = {}

    # Dakwaan
    dakwaan = re.search(
        r'setelah mendengar pembacaan tuntutan pidana(.*?)(?:menimbang)',
        text,
        re.DOTALL
    )

    hasil["dakwaan"] = (
        dakwaan.group(1)[:1000]
        if dakwaan else ""
    )

    # Amar Putusan
    putusan = re.search(
        r'mengadili(.*)',
        text,
        re.DOTALL
    )

    hasil["putusan"] = (
        putusan.group(1)[:1000]
        if putusan else ""
    )

    return hasil

In [7]:
def feature_engineering(text):

    return {
        "jumlah_kata": len(text.split())
    }

In [8]:
rows = []

files = sorted(
    [f for f in os.listdir(RAW_PATH)
     if f.endswith(".txt")]
)

for idx, file in enumerate(tqdm(files), start=1):

    path = os.path.join(RAW_PATH, file)

    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    metadata = extract_metadata(text)

    konten = extract_key_content(text)

    fitur = feature_engineering(text)

    row = {
        "case_id": idx,
        "jenis_perkara": "Pidana Narkotika",
        **metadata,
        **konten,
        **fitur,
        "text_full": text
    }

    rows.append(row)

100%|██████████| 50/50 [00:00<00:00, 282.49it/s]


In [9]:
df = pd.DataFrame(rows)

csv_path = os.path.join(
    OUTPUT_PATH,
    "cases.csv"
)

df.to_csv(
    csv_path,
    index=False,
    encoding="utf-8-sig"
)

print("Jumlah kasus:", len(df))
print("Disimpan ke:", csv_path)

df.head()

Jumlah kasus: 50
Disimpan ke: /content/CBR-Narkotika-PN-Tulungagung/data/processed/cases.csv


,case_id,jenis_perkara,no_perkara,pasal,terdakwa,dakwaan,putusan,jumlah_kata,text_full
0,1,Pidana Narkotika,102 pid sus 2025 pn tlg demi keadilan berdasar...,pasal 148 ; pasal 112 ; pasal 132 ; pasal 193 ...,adila resti sari binti boimain h2,yang diajukan oleh penuntut umum yang pada po...,perkara pidana dengan acara pemeriksaan biasa...,8000,gnomor 102 pid sus 2025 pn tlg demi keadilan b...
1,2,Pidana Narkotika,104 pid sus 2025 pn tlg demi keadilan berdasar...,pasal 112 ; pasal 148 ; pasal 114,zelly yusando alias nsincan bin rismaji 2,,perkara pidana dengan acara pemeriksaan biasa...,7319,direktori utusan mahkamah agung republikp iidn...
2,3,Pidana Narkotika,105 pid sus 2025 pn tlg demi keadilan berdasar...,pasal 148 ; pasal 112 ; pasal 101 ; pasal 13 ;...,riki suhendra bin sumarno 2,yang diajukan oleh penuntut umum ya ng pada p...,perkara pidana dengan agcara pemeriksaan bias...,10522,nomor 105 pid sus 2025 pn tlg demi keadilan be...
3,4,Pidana Narkotika,106 pid sus 2025 pn tlg demi keadilan berdasar...,pasal 148 ; pasal 112 ; pasal 132 ; pasal 193 ...,wiji santoso bin satimin h2,yang diajukan oleh penuntut umum yang pada pe...,perkara pidana dengan acara pemeriksaan biasa...,10152,gnomor 106 pid sus 2025 pn tlg demi keadilan b...
4,5,Pidana Narkotika,107 pid sus 2025 pn tlg demi keadilan berdasar...,pasal 112 ; pasal 127; pasal 132 ; pasal 132; ...,novel saputra ahmad dhani bin sonpan sopyan 2,yang diajukan oleh penuntut umum yang pada po...,perkara pidana dengan acara pemeriksaan biasa...,9426,direktori putusan mahkamah agung re lik indone...


In [10]:
print(df.info())

print("\nContoh Kasus Pertama:\n")

print(df.iloc[0][[
    "case_id",
    "no_perkara",
    "terdakwa",
    "pasal",
    "jumlah_kata"
]])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   case_id        50 non-null     int64 
 1   jenis_perkara  50 non-null     object
 2   no_perkara     50 non-null     object
 3   pasal          50 non-null     object
 4   terdakwa       50 non-null     object
 5   dakwaan        50 non-null     object
 6   putusan        50 non-null     object
 7   jumlah_kata    50 non-null     int64 
 8   text_full      50 non-null     object
dtypes: int64(2), object(7)
memory usage: 3.6+ KB
None

Contoh Kasus Pertama:

case_id                                                        1
no_perkara     102 pid sus 2025 pn tlg demi keadilan berdasar...
terdakwa                       adila resti sari binti boimain h2
pasal          pasal 148 ; pasal 112 ; pasal 132 ; pasal 193 ...
jumlah_kata                                                 8000
Name: 0, dtype: objec

In [11]:
json_path = os.path.join(
    OUTPUT_PATH,
    "cases.json"
)

df.to_json(
    json_path,
    orient="records",
    force_ascii=False,
    indent=4
)

print("JSON tersimpan")

JSON tersimpan


In [14]:
!git add data/processed/cases.csv
!git add data/processed/cases.json

In [26]:
!git status

On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

nothing to commit, working tree clean


In [24]:
!git config --global user.email "aschyprdn@gmail.com"
!git config --global user.name "AsyaPradana"


In [25]:
!git commit -m "Menambahkan hasil Tahap 2 Case Representation"

[main 9692b60] Menambahkan hasil Tahap 2 Case Representation
 2 files changed, 603 insertions(+)
 create mode 100644 data/processed/cases.csv
 create mode 100644 data/processed/cases.json


In [31]:
!git push origin main

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 2 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (6/6), 1.00 MiB | 2.09 MiB/s, done.
Total 6 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 1 local object.
To https://github.com/AsyaPradana/CBR-Narkotika-PN-Tulungagung.git
   cf42125..9692b60  main -> main
